# Runbook 5.2.b: Roles, permissions, and least privilege  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Mitigate Risk · **Maps to:** 5.2.A, 5.2.B · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** Build the audit function a security reviewer would run over a company's role assignments. Given a
map of roles to the permissions they grant, a map of users to their assigned roles, and a map of
users to the minimum permission set their job function actually requires, find every user who
holds at least one permission (via their assigned roles) that is NOT in their job function's
required set.

**Lesson objective.** Explain how permissions are bundled into roles, apply the principle of least privilege to evaluate whether a role assignment is over-permissioned, and identify the risks of privilege creep and excessive standing access.

### What comes in
- `roles`: `dict[str, list[str]]`, role name -> list of permissions it grants.
- `user_roles`: `dict[str, list[str]]`, user name -> list of role names assigned to them.
- `job_requirements`: `dict[str, list[str]]`, user name -> list of permissions their job function
  actually requires.

### What must come out
A dict mapping each violating user to the **sorted** list of their excess (unauthorized)
permissions. Users with no excess permissions are omitted entirely from the result.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```
roles = {"support": ["read_ticket", "write_ticket_reply"], "billing": ["read_invoice", "issue_refund"]}
user_roles = {"dana": ["support", "billing"]}
job_requirements = {"dana": ["read_ticket", "write_ticket_reply"]}

solve(roles, user_roles, job_requirements)
-> {"dana": ["issue_refund", "read_invoice"]}
```

```
roles = {"support": ["read_ticket"]}
user_roles = {"sam": ["support"]}
job_requirements = {"sam": ["read_ticket"]}

solve(roles, user_roles, job_requirements)
-> {}
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] A user with roles granting no permissions beyond their job requirements is omitted from the
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(roles: dict, user_roles: dict, job_requirements: dict) -> dict
```

#### Input
- `roles`: `dict[str, list[str]]`, role name -> permissions it grants.
- `user_roles`: `dict[str, list[str]]`, user name -> assigned role names.
- `job_requirements`: `dict[str, list[str]]`, user name -> permissions their job requires. A user
  absent from this dict requires no permissions (empty set).

#### Output (`result`)
`dict[str, list[str]]` mapping each user who has at least one excess permission to the **sorted**
list of their excess permissions (permissions granted by their roles but not in their job
requirements). Users with zero excess permissions are omitted entirely from the result.

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Permissions from multiple assigned roles are unioned (deduplicated) before comparing against
  job requirements.
- Unknown role names in `user_roles` (not present in `roles`) contribute no permissions.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
from typing import Dict, List


def solve(
    roles: Dict[str, List[str]],
    user_roles: Dict[str, List[str]],
    job_requirements: Dict[str, List[str]],
) -> Dict[str, List[str]]:
    result: Dict[str, List[str]] = {}
    for user, assigned_roles in user_roles.items():
        granted = set()
        for role in assigned_roles:
            granted.update(roles.get(role, []))
        required = set(job_requirements.get(user, []))
        excess = granted - required
        if excess:
            result[user] = sorted(excess)
    return result


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def test_dana_has_excess():
    roles = {"support": ["read_ticket", "write_ticket_reply"], "billing": ["read_invoice", "issue_refund"]}
    user_roles = {"dana": ["support", "billing"]}
    job_requirements = {"dana": ["read_ticket", "write_ticket_reply"]}
    assert impl.solve(roles, user_roles, job_requirements) == {"dana": ["issue_refund", "read_invoice"]}


def test_sam_no_excess_omitted():
    roles = {"support": ["read_ticket"]}
    user_roles = {"sam": ["support"]}
    job_requirements = {"sam": ["read_ticket"]}
    assert impl.solve(roles, user_roles, job_requirements) == {}


def test_multiple_users_mixed():
    roles = {"support": ["read_ticket", "write_ticket_reply"], "admin": ["delete_user", "read_ticket"]}
    user_roles = {"a": ["support"], "b": ["admin"]}
    job_requirements = {"a": ["read_ticket", "write_ticket_reply"], "b": ["read_ticket"]}
    assert impl.solve(roles, user_roles, job_requirements) == {"b": ["delete_user"]}


def test_user_with_no_roles():
    roles = {"support": ["read_ticket"]}
    user_roles = {"ghost": []}
    job_requirements = {"ghost": []}
    assert impl.solve(roles, user_roles, job_requirements) == {}


def test_user_missing_from_job_requirements_treated_as_none_required():
    roles = {"support": ["read_ticket"]}
    user_roles = {"newhire": ["support"]}
    job_requirements = {}
    assert impl.solve(roles, user_roles, job_requirements) == {"newhire": ["read_ticket"]}


def test_overlapping_roles_deduplicated():
    roles = {"r1": ["p1", "p2"], "r2": ["p2", "p3"]}
    user_roles = {"u": ["r1", "r2"]}
    job_requirements = {"u": ["p1", "p2", "p3"]}
    assert impl.solve(roles, user_roles, job_requirements) == {}


def test_empty_inputs():
    assert impl.solve({}, {}, {}) == {}


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Run your finished tool on the examples from the brief, then change one value and predict the answer before you run it again.

- [ ] Every example says OK
- [ ] Your own changed example gives the answer you predicted

In [ ]:
checks = [
    (lambda: solve({}, {}, {}),
     {}),
]

for n, (run, want) in enumerate(checks, 1):
    try:
        got = run()
    except Exception as e:
        got = "error: " + repr(e)
    print("example", n, "OK  " if got == want else "DIFF", "got:", got, " expected:", want)


## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it